# Football Managerial Decision Analytics — Chapter 02
## Match, team and player performance

> **Data.** StatsBomb Open Data (https://github.com/hudl/open-data), provided by StatsBomb / Hudl, used for educational, non-commercial research. Raw data are downloaded from the official repository at run time and are not redistributed here. Every figure carries its chart ID from `docs/VISUALIZATION_CATALOGUE.md`.

This notebook is one chapter of the research record. All logic lives in the shared `src/` package; the cells call it. The same charts are available interactively in the Streamlit dashboard (`streamlit run app.py`).

In [ ]:
# Setup — locate (or fetch) the project code, import the shared package, load the workspace
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/blazingarrows1525/football-managerial-decision-analytics"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "workspace.py").exists()), None)
if ROOT is None:  # opened on its own (e.g. Google Colab): fetch the project code once, next to this notebook
    ROOT = Path.cwd() / "football-managerial-decision-analytics"
    if not (ROOT / "src" / "workspace.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("FMA_PROJECT_ROOT", str(ROOT))  # data cache and outputs live inside the project folder

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.reporting.notebook_tools import (catalogue, explorer, implication, interpret, limitation, show, start,
                                          timed, timings)

pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", "{:.3f}".format)
RUN_ADVANCED_GALLERY = True   # heavier, optional sections; the core story does not depend on them
SEED = 42                             # every resampling and model step takes this seed explicitly

ws = start(seed=SEED)                 # first run downloads and parses; later runs read the processed cache
B, T, FOCUS = ws.bundle, ws.tables, ws.focus
tm = ws.team_matches()                # the focus team's matches in date order
print(f"Advanced sections: {'on' if RUN_ADVANCED_GALLERY else 'off'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

## D · Match, team and player performance

**Research question.** How did the team perform, how much of it is explained by chance quality, and which
players drive it?

**Data and method.** Team match statistics (goals, StatsBomb xG, shots, pass-share possession proxy, field
tilt, PPDA) and player season tables with per-90 rates and role percentiles. Expected points simulate each shot
as a Bernoulli trial with its xG.

In [ ]:
from src.analytics import descriptive as desc
k = desc.season_kpis(B, FOCUS)
xp = ws.xpts(); xp = xp[xp.team_name == FOCUS]
display(pd.DataFrame([{"matches": k["matches"], "W-D-L": f"{k['wins']}-{k['draws']}-{k['losses']}", "points": k["points"],
                       "expected points": round(xp.xpts.sum(), 1), "goals": f"{k['goals_for']}-{k['goals_against']}",
                       "xG": f"{k['xg_for']:.1f}-{k['xg_against']:.1f}", "players used": k["players_used"]}]))
show("L-262", ws)
show("C-042", ws)
gap = k["points"] - xp.xpts.sum()
interpret(f"{FOCUS} took {k['points']:.0f} points from {k['matches']} matches against {xp.xpts.sum():.1f} expected from shot "
          f"quality — {abs(gap):.1f} {'more' if gap > 0 else 'fewer'} than the simulation — and scored {k['goals_for']} "
          f"goals from {k['xg_for']:.1f} xG. Mean xG difference per match: {tm.xg_diff.mean():+.2f}.")
implication("Judge performances on chances created and conceded; the points total alone flatters or punishes a "
            "team for finishing and goalkeeping variance.")
limitation("Expected points are a model output: shots are treated as independent and game state is ignored.")

In [ ]:
show("C-046", ws)
show("C-059", ws)
split = desc.shot_origin_split(B, FOCUS).set_index(["side", "origin"])
f = split.loc["for"]; a = split.loc["against"]
interpret(f"{FOCUS} took {int(f.shots.sum())} shots against {int(a.shots.sum())} for opponents; xG per shot was "
          f"{f.xg.sum() / f.shots.sum():.3f} against {a.xg.sum() / a.shots.sum():.3f}. The gap in **volume** is far larger than "
          f"the gap in **quality**. Possessions that began with a corner, free kick or throw-in produced "
          f"{int(f.loc['Restart-initiated', 'goals'])} of the team's goals from shots.")

In [ ]:
show("C-050", ws)
show("C-064", ws)
st_time = ws.state_time()
tot = st_time[["leading", "drawing", "trailing"]].sum()
z, cv = desc.team_consistency(B, FOCUS)
interpret(f"The team spent {tot.leading / tot.sum():.0%} of its minutes leading, {tot.drawing / tot.sum():.0%} level and "
          f"{tot.trailing / tot.sum():.0%} behind. The most stable match metrics are {cv.index[0]} (CV {cv.iloc[0]:.2f}) and "
          f"{cv.index[1]} ({cv.iloc[1]:.2f}); the most volatile are {cv.index[-1]} ({cv.iloc[-1]:.2f}) and {cv.index[-2]} ({cv.iloc[-2]:.2f}).")
implication("Swings in the volatile metrics are normal match-to-match noise; persistent change in the stable "
            "ones is the earlier warning sign.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("C-043", ws)
    show("C-047", ws)
    show("C-055", ws)
    show("C-063", ws)

### One match in detail

**Question.** What happened, in order, in a representative match? The default is the match with the median xG
difference — neither the best nor the worst.

In [ ]:
mid = ws.default_match()
print(ws.match_label(mid))
show("C-049", ws, match_id=mid)
show("L-248", ws, match_id=mid)
show("L-249", ws, match_id=mid)
limitation("The interval chart is xG per five minutes — a description of when chances occurred, not a validated "
           "momentum measure.")

### Players

**Question.** Who contributes what, relative to players of the same role, and how certain is it?

In [ ]:
sq = ws.squad()
display(sq[["player_name", "primary_position", "minutes_played", "starts", "goals", "xg", "assists", "xa", "xg_p90", "xa_p90",
            "progressive_passes_p90", "benchmark_group", "benchmark_n", "coverage_label"]].head(14))
show("C-044", ws)
show("B-029", ws)
conc = desc.concentration(sq.set_index("player_name").eval("xg + xa"))
interpret(f"The top three contributors ({', '.join(conc.label.head(3))}) account for {conc.share.head(3).sum():.0%} of the "
          f"squad's xG + xA; {int((conc.cumulative_share < 0.8).sum() + 1)} players are needed to reach 80%. "
          f"Gini coefficient {desc.gini(conc.value):.2f}.")
implication("Like-for-like cover and minutes management matter most for the few players who create the chances.")

In [ ]:
pid = ws.default_player()
print("Example player:", ws.player_name(pid))
show("D-069", ws, player_id=pid)
show("D-070", ws, player_id=pid)
show("D-101", ws, player_id=pid)
limitation("Similarity is distance in standardised per-90 space within this dataset; it is not a replacement "
           "recommendation.")

In [ ]:
from src.analytics import uncertainty as unc
reg = sq[(sq.minutes_played >= 900) & (sq.primary_role != "GK")]
ci = unc.bootstrap_rate_intervals(B.pms, reg, "xg", seed=SEED)
show("D-079", ws, metric="xg_p90")
show("D-102", ws)
top, second = ci.iloc[-1], ci.iloc[-2]
sep = "is clearly separated from" if second.hi < top.lo else "overlaps with"
cons = ws.consistency()
steady = cons[cons.prog_mean > cons.prog_mean.median()].sort_values("prog_cv").iloc[0]
interpret(f"{top.player} leads on xG per 90 ({top.rate_p90:.2f}, 95% bootstrap interval {top.lo:.2f}–{top.hi:.2f}), which {sep} "
          f"{second.player} ({second.lo:.2f}–{second.hi:.2f}). Among above-median progressors {steady.player_name} is the "
          f"steadiest (CV {steady.prog_cv:.2f} at {steady.prog_mean:.1f} progressive actions per 90).")
implication("Rank attackers in tiers. Inside overlapping intervals the order is not established by one season.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("L-264", ws)
    show("D-104", ws)
    show("D-106", ws)
    show("B-035", ws)
    show("D-107", ws, player_id=ws.default_player())